# Theo dõi phim và đánh giá

Mẫu GIẢ LẬP seed 42. Học quan hệ nhiều–nhiều và phân tích điểm. Chạy các ô từ trên xuống.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
ROOT=Path.cwd().resolve()
if ROOT.name=="notebooks": ROOT=ROOT.parent
assert (ROOT/"movie_lab").is_dir()
sys.path.insert(0,str(ROOT))
from movie_lab.db import connect
from movie_lab.init_db import initialize
DB=ROOT/"database/movies.db"
if not DB.exists(): initialize(DB,sample=True)

## 1. Đọc bảng và đếm

Một dòng ratings là đánh giá hiện tại của một người cho một phim.

In [ ]:
with connect(DB,True) as conn:
    ratings=pd.read_sql_query("SELECT * FROM ratings",conn)
    movies=pd.read_sql_query("SELECT * FROM movies",conn)
    links=pd.read_sql_query("SELECT * FROM movie_genres",conn)
print("Phim:",len(movies),"Đánh giá:",len(ratings))
display(movies.head(),ratings.head(),links.head())
assert ratings[["movie_id","viewer_id"]].duplicated().sum()==0
assert ratings.score.between(1,10).all()

## 2. LEFT JOIN và phim chưa có đánh giá

Không thay điểm thiếu bằng 0. Trước khi nhìn điểm trung bình, xem số đánh giá.

In [ ]:
from movie_lab.analyze import load_tables
frames=load_tables(DB)
display(frames["movies"])
assert frames["movies"].rating_count.sum()==len(ratings)
assert frames["viewers"].rating_count.sum()==len(ratings)

## 3. JOIN nhiều thể loại

Một phim có nhiều thể loại nên đánh giá xuất hiện ở nhiều nhóm. Không cộng số đánh giá nhóm để tính tổng.

In [ ]:
display(frames["genres"])
print("Đánh giá thật:",len(ratings),"Tổng số trong nhóm thể loại:",frames["genres"].rating_count.sum())

## 4. Vẽ Seaborn

Điểm rời rạc 1–10; histogram cho biết phân phối, không chứng minh chất lượng phim.

In [ ]:
sns.set_theme(style="whitegrid",font="DejaVu Sans")
fig,ax=plt.subplots(figsize=(9,5))
if not ratings.empty: sns.histplot(data=ratings,x="score",discrete=True,ax=ax)
ax.set(title="Phân phối điểm",xlabel="Điểm 1–10",ylabel="Số đánh giá",xticks=range(1,11))
plt.show()

## 5. Xuất kết quả và viết nhận xét

Dùng cùng hàm với terminal. Làm bài tập trước khi đọc lời giải.

In [ ]:
from movie_lab.analyze import analyze
analyze(DB,ROOT/"outputs",min_ratings=5)
print((ROOT/"outputs/report.md").read_text(encoding="utf-8"))

## 6. Tự trả lời

- Ba nhận xét có số liệu?
- Xếp hạng thay đổi thế nào khi tăng ngưỡng số đánh giá?
- Hai giới hạn của dữ liệu giả lập?
- Vì sao so sánh người xem cần biết họ xem phim gì?